In [1]:
   import pandas as pd

   PATH = "s3://sagemaker-salesmesh-myra/raw/Base.csv"
   sample = pd.read_csv(PATH, nrows=5)
   print(sample.columns.tolist())
   sample.head()

['fraud_bool', 'income', 'name_email_similarity', 'prev_address_months_count', 'current_address_months_count', 'customer_age', 'days_since_request', 'intended_balcon_amount', 'payment_type', 'zip_count_4w', 'velocity_6h', 'velocity_24h', 'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w', 'employment_status', 'credit_risk_score', 'email_is_free', 'housing_status', 'phone_home_valid', 'phone_mobile_valid', 'bank_months_count', 'has_other_cards', 'proposed_credit_limit', 'foreign_request', 'source', 'session_length_in_minutes', 'device_os', 'keep_alive_session', 'device_distinct_emails_8w', 'device_fraud_count', 'month']


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,...,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,...,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


In [2]:
df = pd.read_csv(PATH)
print(df.shape)
df.info(memory_usage="deep")

(1000000, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 32 columns):
 #   Column                            Non-Null Count    Dtype  
---  ------                            --------------    -----  
 0   fraud_bool                        1000000 non-null  int64  
 1   income                            1000000 non-null  float64
 2   name_email_similarity             1000000 non-null  float64
 3   prev_address_months_count         1000000 non-null  int64  
 4   current_address_months_count      1000000 non-null  int64  
 5   customer_age                      1000000 non-null  int64  
 6   days_since_request                1000000 non-null  float64
 7   intended_balcon_amount            1000000 non-null  float64
 8   payment_type                      1000000 non-null  object 
 9   zip_count_4w                      1000000 non-null  int64  
 10  velocity_6h                       1000000 non-null  float64
 11  velocity_24h            

In [3]:
print("Fraud rate:", round(df["fraud_bool"].mean() * 100, 3), "%")
df.groupby("month")["fraud_bool"].agg(["count", "mean"])

Fraud rate: 1.103 %


,count,mean
month,,
0,132440,0.011326
1,127620,0.009387
2,136979,0.008746
3,150936,0.009222
4,127691,0.011371
5,119323,0.011825
6,108168,0.013405
7,96843,0.014746


In [4]:
num_cols = df.select_dtypes("number").columns
neg = (df[num_cols] < 0).sum()
neg[neg > 0].sort_values(ascending=False)

intended_balcon_amount          742523
prev_address_months_count       712920
bank_months_count               253635
credit_risk_score                14445
current_address_months_count      4254
session_length_in_minutes         2015
device_distinct_emails_8w          359
velocity_6h                         44
dtype: int64

In [5]:
for col in ["source", "device_os", "payment_type", "employment_status", "housing_status"]:
    print(df[col].value_counts(), "\n")

source
INTERNET    992952
TELEAPP       7048
Name: count, dtype: int64 

device_os
other        342728
linux        332712
windows      263506
macintosh     53826
x11            7228
Name: count, dtype: int64 

payment_type
AB    370554
AA    258249
AC    252071
AD    118837
AE       289
Name: count, dtype: int64 

employment_status
CA    730252
CB    138288
CF     44034
CC     37758
CD     26522
CE     22693
CG       453
Name: count, dtype: int64 

housing_status
BC    372143
BB    260965
BA    169675
BE    169135
BD     26161
BF      1669
BG       252
Name: count, dtype: int64 



In [6]:
df.nunique().sort_values().head(10)

device_fraud_count           1
fraud_bool                   2
keep_alive_session           2
foreign_request              2
phone_home_valid             2
has_other_cards              2
email_is_free                2
source                       2
phone_mobile_valid           2
device_distinct_emails_8w    4
dtype: int64

In [7]:
KEEP = ["name_email_similarity", "customer_age", "prev_address_months_count",
        "current_address_months_count", "zip_count_4w", "velocity_6h", "velocity_24h",
        "velocity_4w", "date_of_birth_distinct_emails_4w", "email_is_free",
        "phone_home_valid", "phone_mobile_valid", "foreign_request",
        "session_length_in_minutes", "keep_alive_session", "device_distinct_emails_8w"]

df.groupby("fraud_bool")[KEEP].mean().T

fraud_bool,0,1
name_email_similarity,0.494815,0.393161
customer_age,33.609125,40.858645
prev_address_months_count,16.839647,5.861365
current_address_months_count,86.273232,114.801161
zip_count_4w,1572.138693,1622.311542
velocity_6h,5670.664988,5183.913444
velocity_24h,4771.528849,4613.138798
velocity_4w,4857.444566,4755.844185
date_of_birth_distinct_emails_4w,9.526521,7.443195
email_is_free,0.528423,0.661075
